In [1]:
import json
import re
import pandas as pd

from tqdm import tqdm

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

In [2]:
# -------------------------
# Load fine-tuned model
# -------------------------
model_name = "madox81/SmolLM2-Cyber-Insight"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

config.json:   0%|          | 0.00/960 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/671 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

chat_template.jinja:   0%|          | 0.00/398 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49153, 2048, padding_idx=49152)
    (layers): ModuleList(
      (0-23): 24 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (v_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,)

In [3]:
# Model Generator
def generator(messages):
    # Apply Chat Template
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=True,
        return_tensors='pt',
        add_generation_prompt=True
    ).to(model.device)

    # Generation
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            temperature=None,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # Decode
    prompt_length = inputs['input_ids'].shape[1]
    generated_tokens = output[0][prompt_length:]
    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )

    return response.strip()


In [4]:
# -------------------------
# Load dataset
# -------------------------

dataset = load_dataset("madox81/mittre_severity_ds", split="test")

# -------------------------
# Containers
# -------------------------

true_tactics = []
pred_tactics = []

true_techniques = []
pred_techniques = []

true_severity = []
pred_severity = []

true_risk = []
pred_risk = []

# -------------------------
# JSON extraction helper
# -------------------------

def extract_json(text):

    try:
        match = re.search(r"\{.*\}", text, re.DOTALL)

        if match:
            return json.loads(match.group())

    except:
        pass

    return {}


README.md:   0%|          | 0.00/554 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/114k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/17.4k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2925 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/352 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/170 [00:00<?, ? examples/s]

In [5]:
# -------------------------
# Evaluation loop
# -------------------------

for i in tqdm(range(len(dataset)), desc="Samples"):

    messages = dataset[i]["messages"]

    prompt = [messages[0]]
    ground_truth = json.loads(messages[1]["content"])

    output = generator(prompt)

    prediction = extract_json(output)

    # -------------------------
    # MITRE task
    # -------------------------

    if "tactics" in ground_truth:

        true_tactics.append(ground_truth["tactics"])
        true_techniques.append(ground_truth["techniques"])

        pred_tactics.append(prediction.get("tactics", []))
        pred_techniques.append(prediction.get("techniques", []))

    # -------------------------
    # Severity task
    # -------------------------

    if "severity" in ground_truth:

        true_severity.append(ground_truth["severity"])
        pred_severity.append(prediction.get("severity", "Unknown"))

        true_risk.append(ground_truth["business_risk"])
        pred_risk.append(prediction.get("business_risk", "Unknown"))

# -------------------------
# MITRE Evaluation
# -------------------------

mlb_tactic = MultiLabelBinarizer()
mlb_tech = MultiLabelBinarizer()

true_tactic_bin = mlb_tactic.fit_transform(true_tactics)
pred_tactic_bin = mlb_tactic.transform(pred_tactics)

true_tech_bin = mlb_tech.fit_transform(true_techniques)
pred_tech_bin = mlb_tech.transform(pred_techniques)

tactic_precision = precision_score(true_tactic_bin, pred_tactic_bin, average="macro", zero_division=0)
tactic_recall = recall_score(true_tactic_bin, pred_tactic_bin, average="macro", zero_division=0)
tactic_f1 = f1_score(true_tactic_bin, pred_tactic_bin, average="macro", zero_division=0)

tech_precision = precision_score(true_tech_bin, pred_tech_bin, average="macro", zero_division=0)
tech_recall = recall_score(true_tech_bin, pred_tech_bin, average="macro", zero_division=0)
tech_f1 = f1_score(true_tech_bin, pred_tech_bin, average="macro", zero_division=0)

# -------------------------
# Severity evaluation
# -------------------------

sev_precision = precision_score(true_severity, pred_severity, average="macro", zero_division=0)
sev_recall = recall_score(true_severity, pred_severity, average="macro", zero_division=0)
sev_f1 = f1_score(true_severity, pred_severity, average="macro", zero_division=0)

# -------------------------
# Business risk evaluation
# -------------------------

risk_precision = precision_score(true_risk, pred_risk, average="macro", zero_division=0)
risk_recall = recall_score(true_risk, pred_risk, average="macro", zero_division=0)
risk_f1 = f1_score(true_risk, pred_risk, average="macro", zero_division=0)

# -------------------------
# Final evaluation table
# -------------------------

results = pd.DataFrame({
    "Task":[
        "Tactic Classification",
        "Technique Classification",
        "Severity Classification",
        "Business Risk Classification"
        ],
    "Precision":[
        tactic_precision,
        tech_precision,
        sev_precision,
        risk_precision
    ],
    "Recall":[
        tactic_recall,
        tech_recall,
        sev_recall,
        risk_recall
    ],
    "F1":[
        tactic_f1,
        tech_f1,
        sev_f1,
        risk_f1
    ],
})


Samples: 100%|██████████| 170/170 [03:12<00:00,  1.13s/it]
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['Credentials', 'Data Destruction'] will be ignored
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['Access Token Manipulation', 'Address Resolution Protocol (ARP) Discovery', 'DNS Amplification Tool', 'DNS Spoofing', 'Data Encrypted for Transmission Over Unsecured Protocols', 'Docker', 'Domain Policy Modification', 'Exfiltration Over Web Service', 'External Hardware Insertion', 'File Transfer', 'File Transfer Protocols', 'Hijack Execution Flow', 'Ingress Tool Transfer', 'Network Drive', 'Network Flow Analysis', 'PowerShell', 'Process Injection', 'Proxy', 'Remote Service Session Hijacking', 'Scheduled Task/Job', 'Security Information Management Software (SIMS)', 'Server Software Discovery', 'Unsecured Credentials', 'Use Alternate Authentication 

In [6]:
print("\n==============================")
print("FINAL MODEL EVALUATION")
print("==============================")

results


FINAL MODEL EVALUATION


,Task,Precision,Recall,F1
0,Tactic Classification,0.890246,0.796610,0.824079
1,Technique Classification,0.443333,0.454103,0.432510
2,Severity Classification,0.579365,0.570794,0.571659
3,Business Risk Classification,1.000000,1.000000,1.000000
